## Install dependencies


In [3]:
# Run once
!pip3 install statsbombpy requests beautifulsoup4 fuzzywuzzy python-Levenshtein lxml pandas numpy soccerdata 

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 64 kB 2.4 MB/s eta 0:00:01
     |████████████████████████████████| 109 kB 5.8 MB/s eta 0:00:01
     |████████████████████████████████| 8.6 MB 13.6 MB/s eta 0:00:01     |████████████▌                   | 3.3 MB 13.6 MB/s eta 0:00:01
     |████████████████████████████████| 10.8 MB 13.9 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 17.7 MB/s eta 0:00:01
     |████████████████████████████████| 55 kB 11.0 MB/s eta 0:00:01
     |████████████████████████████████| 70 kB 14.2 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 11.0 MB/s eta 0:00:01
     |████████████████████████████████| 305 kB 14.8 MB/s eta 0:00:01
     |████████████████████████████████| 133 kB 12.4 MB/s eta 0:00:01
     |████████████████████████████████| 65 kB 9.0 MB/s  eta 0:00:01
     |████████████████████████████████| 131 kB 11.5 MB/s eta 0:00:01
     |█████████████████████████

## Imports


In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import warnings
warnings.filterwarnings('ignore')
from bs4 import BeautifulSoup
from fuzzywuzzy import process, fuzz
from rapidfuzz import process, fuzz
from statsbombpy import sb

## STEP 1 — Get all unique players from La Liga data


In [24]:
COMP_ID, SEASON_ID = 11, 90
matches = sb.matches(competition_id=COMP_ID, season_id=SEASON_ID)
print(matches.head(10).to_string(index=False))
all_events = []
for mid in matches.match_id:
    ev = sb.events(match_id=mid)
    ev['match_id'] = mid
    all_events.append(ev[ev.type == 'Pass'][['player', 'player_id']].drop_duplicates())

players_df = (pd.concat(all_events)
              .drop_duplicates('player_id')
              .reset_index(drop=True)
              .sort_values('player'))

print(f"Unique players to look up: {len(players_df)}")
print(players_df.head(10).to_string(index=False))

 match_id match_date     kick_off  home_score  away_score match_status match_status_360               last_updated           last_updated_360  match_week  competition_id competition_country_name competition_name     competition  season_id    season  home_team_id        home_team home_team_gender home_team_group  home_team_country_id home_team_country_name  away_team_id   away_team away_team_gender away_team_group  away_team_country_id away_team_country_name  competition_stage_id competition_stage  stadium_id                    stadium  stadium_country_id stadium_country_name  referee_id                      referee  referee_country_id referee_country_name                    home_managers                  away_managers  home_manager_id                home_manager_name home_manager_nickname home_manager_dob  home_manager_country_id home_manager_country_name  away_manager_id              away_manager_name away_manager_nickname away_manager_dob  away_manager_country_id away_manager_country

## APPROACH — FIFA 21 dataset (fastest — download from Kaggle)


In [26]:
# Download from: https://www.kaggle.com/datasets/stefanoleone992/fifa-21-complete-player-dataset
# File: players_21.csv  (~45MB)
# No scraping needed — just download and point to the path below
# Fifa 21 dataset and StatsBomb player names are not identical, so we need to do some fuzzy matching to get the preferred foot for each player.

ALIASES = {

    "Moriba Kourouma Kourouma": "Ilaix Moriba",
    "Rodrigo Sánchez Rodriguez": "Rodri Sanchez",
    "Gabriel Veiga Novas": "Gabri Veiga",
    "Antonio Barragán Fernández": "Barragan",
    "Yeremi Jesús Pino Santos": "Yeremy Pino",

} 

#Normalise names to match FIFA dataset
import unicodedata

def strip_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )

def load_fifa_foot(fifa_csv_path='players_21.csv'):
    """Load preferred foot from FIFA 21 dataset."""
    fifa = pd.read_csv(fifa_csv_path,
                       usecols=['short_name', 'long_name', 'preferred_foot'],
                       low_memory=False)
    fifa = fifa.drop_duplicates('long_name').copy()
    print(f"FIFA 21: {len(fifa)} players loaded")
    print(f"Foot distribution:\n{fifa['preferred_foot'].value_counts()}")
    return fifa


def match_fifa_to_statsbomb(statsbomb_players, fifa_df, threshold=80):
    """Fuzzy name match StatsBomb players to FIFA dataset."""
    fifa_names = fifa_df['long_name'].tolist()
    matched    = []

    for _, row in statsbomb_players.iterrows():
        sb_name = row['player']

        # Apply aliases
        if sb_name in ALIASES:
            sb_name = ALIASES[sb_name]
        # 1. Exact match
        exact = fifa_df[fifa_df['long_name'].str.lower() == sb_name.lower()]
        if len(exact) > 0:
            foot = exact.iloc[0]['preferred_foot']
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': foot, 'match_score': 100,
                            'matched_name': exact.iloc[0]['long_name']})
            continue

        # 2. Fuzzy match
        best_match, score = process.extractOne(
            sb_name, fifa_names, scorer=fuzz.token_sort_ratio)
        if score >= threshold:
            foot = fifa_df[fifa_df['long_name'] == best_match].iloc[0]['preferred_foot']
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': foot, 'match_score': score,
                            'matched_name': best_match})
        else:
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': None, 'match_score': score,
                            'matched_name': best_match})

    df = pd.DataFrame(matched)
    matched_n = df['preferred_foot'].notna().sum()
    print(f"\nMatched: {matched_n}/{len(df)} players ({matched_n/len(df):.1%})")
    print(f"Unmatched players: {df[df.preferred_foot.isna()]['player'].tolist()}")
    print(f"\nLow-confidence matches (score < 90):")
    low = df[(df.match_score < 90) & df.preferred_foot.notna()]
    print(low[['player','matched_name','match_score','preferred_foot']].to_string(index=False))
    return df

# ── Run after downloading players_21.csv ──
fifa_df = load_fifa_foot('players_21.csv')
foot_df = match_fifa_to_statsbomb(players_df, fifa_df)
foot_df.to_csv('preferred_foot_fifa.csv', index=False)

FIFA 21: 18896 players loaded
Foot distribution:
preferred_foot
Right    14409
Left      4487
Name: count, dtype: int64

Matched: 358/379 players (94.5%)
Unmatched players: ['Amankwaa Akurugu', 'Barragan', 'Carlos Domínguez Cáceres', 'Clément Lenglet', 'Emiliano Ariel Rigoni', 'Florian Grégoire Claude Lejeune', 'Gabri Veiga', 'José Antonio Miranda Boacho', 'Juan Antonio Iglesias Sánchez', 'Ilaix Moriba', 'Nehuén Pérez', 'Robert Navarro Muñoz', 'Robin Aime Robert Le Normand', 'Rodri Sanchez', 'Sergino Dest', 'Shinji Okazaki', 'Takefusa Kubo', 'Yann Bodiger', 'Yeremy Pino', 'Yunus Dimoara Musah', 'Óscar Mingueza García']

Low-confidence matches (score < 90):
                  player                      matched_name  match_score preferred_foot
Carlos Henrique Casimiro Carlos Henrique Venancio Casimiro           84          Right
 Francis Joseph Coquelin                  Francis Coquelin           82          Right
    Giorgi Kochorashvili               Giorgi Kharaishvili           87   

## APPROACH D — Wikidata SPARQL (free, open, citable)


In [ ]:
def fetch_wikidata_foot():
    """
    Fetch preferred foot for all footballers from Wikidata public SPARQL.
    Wikidata property P2354 = preferred foot.
    Completely free, open, and academically citable.
    """
    endpoint = "https://query.wikidata.org/sparql"
    query = """
    SELECT DISTINCT ?playerLabel ?foot ?footLabel WHERE {
      ?player wdt:P106 wd:Q937857 .   # occupation: association football player
      ?player wdt:P2354 ?foot .        # preferred foot
      SERVICE wikibase:label {
        bd:serviceParam wikibase:language "en" .
      }
    }
    """
    r = requests.get(
        endpoint,
        params={'query': query, 'format': 'json'},
        headers={
            'User-Agent': 'xPassResearch/1.0 (academic research)',
            'Accept':     'application/sparql-results+json',
        },
        timeout=60
    )
    if r.status_code != 200:
        print(f"Wikidata error {r.status_code}: {r.text[:200]}")
        return pd.DataFrame()

    results = r.json()['results']['bindings']
    rows = [
        {'player_wiki':    p['playerLabel']['value'],
         'preferred_foot': p['footLabel']['value'].replace(' foot', '').title()}
        for p in results
    ]
    df = pd.DataFrame(rows).drop_duplicates('player_wiki')
    print(f"Wikidata: {len(df)} footballers with preferred foot")
    print(df['preferred_foot'].value_counts())
    return df


def match_wikidata_to_statsbomb(statsbomb_players, wikidata_df, threshold=85):
    """Fuzzy match Wikidata player names to StatsBomb names."""
    wiki_names = wikidata_df['player_wiki'].tolist()
    matched    = []

    for _, row in statsbomb_players.iterrows():
        sb_name     = row['player']
        best, score = process.extractOne(sb_name, wiki_names, scorer=fuzz.token_sort_ratio)
        if score >= threshold:
            foot = wikidata_df[wikidata_df['player_wiki'] == best].iloc[0]['preferred_foot']
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': foot, 'score': score, 'matched': best})
        else:
            matched.append({'player_id': row['player_id'], 'player': sb_name,
                            'preferred_foot': None, 'score': score, 'matched': best})

    df = pd.DataFrame(matched)
    n  = df.preferred_foot.notna().sum()
    print(f"Matched: {n}/{len(df)} players ({n/len(df):.1%})")
    return df

# ── Run ──
# wiki_df   = fetch_wikidata_foot()
# foot_wiki = match_wikidata_to_statsbomb(players_df, wiki_df)
# foot_wiki.to_csv('preferred_foot_wikidata.csv', index=False)

## STEP 2 — Validate: inferred vs ground-truth preferred foot


In [ ]:
def validate_inference(passes_with_inferred, foot_ground_truth_df):
    """
    Compare your usage-inferred preferred foot against ground truth.
    The accuracy number is itself a finding worth reporting.
    """
    # Inferred: from Cell 4 in your main notebook (pass usage distribution)
    inferred = (passes_with_inferred[['player_id', 'player', 'preferred_foot']]
                .drop_duplicates('player_id')
                .rename(columns={'preferred_foot': 'inferred'}))

    # Ground truth: from whichever approach above returned data
    ground   = (foot_ground_truth_df[['player_id', 'preferred_foot']]
                .rename(columns={'preferred_foot': 'ground_truth'}))

    merged   = inferred.merge(ground, on='player_id', how='inner')
    valid    = merged[merged['inferred'].isin(['Right', 'Left'])].copy()
    correct  = (valid['inferred'] == valid['ground_truth']).sum()
    accuracy = correct / len(valid) if len(valid) > 0 else 0

    print(f"\n── Foot Inference Validation ──")
    print(f"Players compared : {len(valid)}")
    print(f"Accuracy         : {accuracy:.1%}")

    mismatches = valid[valid['inferred'] != valid['ground_truth']]
    if len(mismatches):
        print(f"\nMismatches ({len(mismatches)} players):")
        print(mismatches[['player', 'inferred', 'ground_truth']].to_string(index=False))
    else:
        print("\nNo mismatches — inference was perfect on this sample.")

    return accuracy

# Usage (after running one of the approaches above):
accuracy = validate_inference(passes, foot_df)

## STEP 3 — Merge ground-truth foot back into passes and re-run model


In [ ]:
def merge_real_foot(passes_df, foot_df, foot_col='preferred_foot'):
    """
    Replace usage-inferred weak_foot with ground-truth preferred foot.
    foot_df must have: player_id, preferred_foot ('Right' / 'Left' / 'Both')
    After calling this, re-run Cell 5 onwards in your main notebook.
    """
    drop_cols    = ['preferred_foot', 'total', 'weak_foot']
    passes_clean = passes_df.drop(columns=[c for c in drop_cols if c in passes_df.columns])

    passes_clean = passes_clean.merge(
        foot_df[['player_id', foot_col]].rename(columns={foot_col: 'preferred_foot'}),
        on='player_id', how='left'
    )

    def flag_weak_real(row):
        if row['pass_body_part'] not in ('Right Foot', 'Left Foot'):
            return np.nan
        if row.get('preferred_foot') not in ('Right', 'Left'):
            return np.nan
        used = 'Right' if row['pass_body_part'] == 'Right Foot' else 'Left'
        return int(used != row['preferred_foot'])

    passes_clean['weak_foot'] = passes_clean.apply(flag_weak_real, axis=1)

    # Coverage report
    total   = passes_clean['pass_body_part'].isin(['Right Foot', 'Left Foot']).sum()
    covered = (passes_clean[passes_clean['pass_body_part']
               .isin(['Right Foot','Left Foot'])]['preferred_foot'].notna().sum())

    print(f"\nGround-truth coverage: {covered:,}/{total:,} foot passes ({covered/total:.1%})")
    print(f"Weak foot distribution: {passes_clean['weak_foot'].value_counts(dropna=False).to_dict()}")
    print("\nNow re-run Cell 5 onwards in laliga_360_xpass.ipynb")
    return passes_clean

# ── Full workflow example ──
# Choose ONE of the approaches above, then:
#
# Option A (FBref):
#   passes = merge_real_foot(passes, foot_fbref)
#
# Option B (Transfermarkt):
#   foot_tm = scrape_transfermarkt(players_df)
#   passes  = merge_real_foot(passes, foot_tm)
#
# Option C (FIFA 21 — recommended):
#   fifa_df = load_fifa_foot('players_21.csv')
#   foot_df = match_fifa_to_statsbomb(players_df, fifa_df)
#   passes  = merge_real_foot(passes, foot_df)
#
# Option D (Wikidata):
#   wiki_df   = fetch_wikidata_foot()
#   foot_wiki = match_wikidata_to_statsbomb(players_df, wiki_df)
#   passes    = merge_real_foot(passes, foot_wiki)

## What to expect — results comparison

## What to Expect After Switching to Ground-Truth Foot

| Metric | Inferred foot | Ground-truth foot |
|---|---|---|
| Coverage | ~80% of players | ~90-95% (FIFA/TM) |
| Weak foot accuracy | ~85-90% | 100% by definition |
| Cross miscalibration gap | −14.8pp | Likely larger (bias was understated) |
| Overall AUC change | — | Minimal (+0.001 at most) |

**Key insight:** if the inference was 85% accurate, ~15% of weak-foot passes were
labelled as strong-foot and vice versa. This dilutes the calibration signal.
The real gap will be **larger** than −14.8pp, strengthening your finding.

**Validation accuracy is itself a contribution:** reporting "our usage-based inference
was X% accurate against FIFA ground truth" is a methodological result that
tells future researchers how much they can trust this shortcut when explicit
foot data is unavailable.
